# Notebook 1: Data Exploration

**Telecom Churn Case Study**

This notebook covers:
- Loading and validating the raw telecom dataset
- Schema inspection (shape, dtypes, memory usage)
- Missing value analysis
- Univariate distributions for key features
- Temporal patterns across the 4-month window

**Dataset:** 99,999 rows × 226 columns, June–September 2014

In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
sys.path.insert(0, str(Path().resolve().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_DIR, TELECOM_DATA_FILE, MONTHS
from src.data_loader import (
    load_telecom_data,
    validate_data,
    get_column_types,
    get_missing_value_summary,
)

sns.set_style('whitegrid')
sns.set_palette('tab10')
plt.rcParams['figure.dpi'] = 100

print('Setup complete.')

## 1. Load Data

Place `telecom_churn_data.csv` in `data/raw/` before running this cell.

In [ ]:
# Load dataset
df = load_telecom_data()

print(f'Shape: {df.shape}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
df.head(3)

## 2. Schema Validation

In [ ]:
# Validate schema
validate_data(df)

# Column type breakdown
date_cols, id_cols, num_cols = get_column_types(df)
print(f'Date columns:       {len(date_cols)}')
print(f'ID columns:         {len(id_cols)}')
print(f'Numeric columns:    {len(num_cols)}')
print(f'Other columns:      {df.shape[1] - len(date_cols) - len(id_cols) - len(num_cols)}')

In [ ]:
# Count columns per month
for month in MONTHS:
    cols = [c for c in df.columns if c.endswith(f'_{month}')]
    print(f'Month {month}: {len(cols)} columns')

## 3. Data Types & Basic Stats

In [ ]:
# Data types summary
df.dtypes.value_counts()

In [ ]:
# Numeric summary for key columns
key_cols = ['arpu_6', 'arpu_7', 'arpu_8', 'arpu_9']
df[key_cols].describe().round(2)

## 4. Missing Value Analysis

In [ ]:
# Missing value summary
missing_summary = get_missing_value_summary(df)
print(f'Columns with missing values: {len(missing_summary)}')
missing_summary.head(20)

In [ ]:
# Visualize missing value distribution
fig, ax = plt.subplots(figsize=(12, 4))
missing_summary['missing_pct'].hist(bins=30, ax=ax, color='#F44336', alpha=0.7)
ax.axvline(x=70, color='black', linestyle='--', label='70% threshold')
ax.set_title('Distribution of Missing Value Percentages Across Columns', fontsize=13)
ax.set_xlabel('Missing %')
ax.set_ylabel('Number of Columns')
ax.legend()
plt.tight_layout()
plt.show()

## 5. ARPU Distribution Across Months

In [ ]:
# ARPU distribution across all 4 months
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
month_names = {'6': 'June', '7': 'July', '8': 'August', '9': 'September'}

for i, month in enumerate(MONTHS):
    ax = axes[i // 2][i % 2]
    col = f'arpu_{month}'
    if col in df.columns:
        data = df[col].dropna()
        data = data[data > 0]  # exclude zeros for better visualization
        ax.hist(data, bins=50, color=f'C{i}', alpha=0.75, edgecolor='white')
        ax.set_title(f'ARPU Distribution — {month_names[month]}', fontsize=12)
        ax.set_xlabel('ARPU (INR)')
        ax.set_ylabel('Frequency')
        ax.axvline(data.median(), color='black', linestyle='--', label=f'Median: {data.median():.0f}')
        ax.legend()

plt.suptitle('ARPU Distribution by Month (excluding zeros)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Call Minutes Distribution

In [ ]:
# Total call minutes (incoming + outgoing) per month
fig, ax = plt.subplots(figsize=(12, 5))

for i, month in enumerate(MONTHS):
    ic_col = f'total_ic_mou_{month}'
    og_col = f'total_og_mou_{month}'
    if ic_col in df.columns and og_col in df.columns:
        total = (df[ic_col].fillna(0) + df[og_col].fillna(0))
        ax.hist(total[total > 0], bins=50, alpha=0.5, label=f'Month {month} ({month_names[month]})', color=f'C{i}')

ax.set_title('Total Call Minutes Distribution by Month', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Call Minutes')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Data Usage Distribution

In [ ]:
# 2G + 3G data usage per month
fig, ax = plt.subplots(figsize=(12, 5))

for i, month in enumerate(MONTHS):
    v2g = f'vol_2g_mb_{month}'
    v3g = f'vol_3g_mb_{month}'
    if v2g in df.columns and v3g in df.columns:
        total = (df[v2g].fillna(0) + df[v3g].fillna(0))
        ax.hist(total[total > 0], bins=50, alpha=0.5, label=f'Month {month} ({month_names[month]})', color=f'C{i}')

ax.set_title('Total Data Usage Distribution by Month (MB)', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Data Usage (MB)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

Key observations:
1. **Missing values** are concentrated in a small subset of columns, mostly data-related features for customers who never used data services.
2. **ARPU distributions** are right-skewed, with median ARPU declining from August to September (churn precursor).
3. **Call minutes** show a clear bimodal pattern — heavy users vs. minimal users.
4. **Data usage** is highly skewed; many customers use zero data (voice-only subscribers).

➡️ Continue to `02_detailed_analysis.ipynb` for high-value filtering, churn tagging, and bivariate analysis.